# einops.einsum — procedural drill

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-einsum`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five `einops.einsum` patterns that ramp from elementwise product → matrix multiply → omit-to-reduce → batched matmul → attention QK^T. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import einsum

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Deep Learning` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `einops-einsum`**, which bridges to the bank subtopic `Einops: Deep Learning` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-einsum"
DD_SUBTOPIC = "Einops: Deep Learning"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## einops.einsum — quick refresher

`einsum(*tensors, pattern)` performs sum-contraction over named indices:
1. **Elementwise** — `'i j, i j -> i j'` multiplies pointwise (no reduction).
2. **Matmul** — `'i k, k j -> i j'` contracts the shared `k` (sum-reduce).
3. **Single-operand reduce** — `'i j -> i'` sums over `j` (no second tensor needed).
4. **Batched** — `'b i k, b k j -> b i j'` carries `b` through, contracts `k`.

**The two rules:**
- An index that appears on input AND output → preserved (broadcast-like).
- An index that appears on input but NOT on output → sum-contracted.

### Exercise 1 — elementwise product (Hadamard)

> ```yaml
> Difficulty: ⚪⚪⚪⚪⚪
> Bloom level: Remember
> LO: Recall that an einsum pattern with every index on every side performs elementwise multiplication.
> Keywords: elementwise, hadamard, no-contraction
> ```

**KCs targeted:** `einsum-elementwise`

Implement `ex1_hadamard(x, y)` to compute the elementwise product of two 2-D tensors of the same shape.

Input shapes: both `(i, j)`. Output shape: `(i, j)`.

Use `einops.einsum(...)` — not `x * y`. The point is to write the pattern. Every index that appears on the output must also appear in every input — no contraction, no broadcast.

In [ ]:
def ex1_hadamard(x: Tensor, y: Tensor) -> Tensor:
    """Elementwise product of two (i, j) tensors via einsum."""
    raise NotImplementedError()


def _test_ex1():
    x = t.arange(12).reshape(3, 4).float()
    y = t.arange(12, 24).reshape(3, 4).float()
    z = ex1_hadamard(x, y)
    assert z.shape == (3, 4), f'expected (3,4), got {z.shape}'
    assert t.equal(z, x * y), 'values differ from x * y'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_hadamard(x: Tensor, y: Tensor) -> Tensor:
    return einsum(x, y, 'i j, i j -> i j')
```
</details>

### Exercise 2 — matrix multiplication (single-index contraction)

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Bloom level: Apply
> LO: Apply the einsum convention to perform matrix multiplication by contracting the shared inner index.
> Keywords: matmul, contraction, shared-index
> ```

**KCs targeted:** `einsum-matmul-contraction`

Implement `ex2_matmul(x, y)` for the standard matrix product.

Input shapes: `x` is `(i, k)`, `y` is `(k, j)`. Output shape: `(i, j)`.

The inner index `k` appears in both inputs but **not** on the output — that's einsum's signal to sum-contract over it. Equivalent to `x @ y`.

In [ ]:
def ex2_matmul(x: Tensor, y: Tensor) -> Tensor:
    """Matrix multiply (i, k) @ (k, j) → (i, j) via einsum."""
    raise NotImplementedError()


def _test_ex2():
    x = t.arange(2 * 3).reshape(2, 3).float()
    y = t.arange(3 * 4).reshape(3, 4).float()
    z = ex2_matmul(x, y)
    assert z.shape == (2, 4), f'expected (2,4), got {z.shape}'
    assert t.allclose(z, x @ y), 'values differ from x @ y'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_matmul(x: Tensor, y: Tensor) -> Tensor:
    return einsum(x, y, 'i k, k j -> i j')
```

**Why does `k` disappear?** The einsum rule: every axis name that appears on input but **not** on output is sum-reduced. `k` appears in both `x` and `y` (so values get multiplied pointwise along that axis), and the absence on the output side triggers the sum. Net result: `z[i, j] = sum_k x[i, k] * y[k, j]`.
</details>

### Exercise 3 — row sum (omit-to-reduce)

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the omit-to-reduce convention with a single operand to compute a row sum.
> Keywords: reduction, omit-axis, single-operand
> ```

**KCs targeted:** `einsum-reduce-via-omit`

Implement `ex3_row_sum(x)` to sum each row of a 2-D tensor.

Input shape: `(i, j)`. Output shape: `(i,)`.

Use `einops.einsum` with a **single** operand. The axis you want to reduce just gets dropped from the output side of the pattern — there's no `sum` keyword.

Equivalent to `x.sum(dim=1)`.

In [ ]:
def ex3_row_sum(x: Tensor) -> Tensor:
    """Sum each row. (i, j) → (i,) via einsum (single operand)."""
    raise NotImplementedError()


def _test_ex3():
    x = t.arange(3 * 4).reshape(3, 4).float()
    z = ex3_row_sum(x)
    assert z.shape == (3,), f'expected (3,), got {z.shape}'
    assert t.allclose(z, x.sum(dim=1)), 'values differ from x.sum(dim=1)'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_row_sum(x: Tensor) -> Tensor:
    return einsum(x, 'i j -> i')
```

**Single-operand einsum** is just the reduction case of the general convention: any axis name on the input that's missing from the output is summed. `'i j -> i'` sums over `j`. `'i j ->'` (empty output) would sum both axes and return a scalar.
</details>

### Exercise 4 — batched matrix multiply (preserve a batch axis)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply einsum to batched matmul by carrying a non-contracted axis through both operands and the output.
> Keywords: batched, matmul, preserve-axis
> ```

**KCs targeted:** `einsum-batched`

Implement `ex4_batched_matmul(x, y)` for a batched matmul.

Input shapes: `x` is `(b, i, k)`, `y` is `(b, k, j)`. Output shape: `(b, i, j)`.

The `b` axis appears in both inputs **and** on the output — that's how einsum carries it through. The `k` axis still contracts. Equivalent to `torch.bmm(x, y)` or `x @ y` (PyTorch broadcasts the last two dims).

In [ ]:
def ex4_batched_matmul(x: Tensor, y: Tensor) -> Tensor:
    """Batched matmul (b, i, k) @ (b, k, j) → (b, i, j) via einsum."""
    raise NotImplementedError()


def _test_ex4():
    x = t.arange(2 * 3 * 4).reshape(2, 3, 4).float()
    y = t.arange(2 * 4 * 5).reshape(2, 4, 5).float()
    z = ex4_batched_matmul(x, y)
    assert z.shape == (2, 3, 5), f'expected (2,3,5), got {z.shape}'
    assert t.allclose(z, x @ y), 'values differ from x @ y'
    _dd_passed.add('ex4')
    print("ex4 ✓")

_test_ex4()

<details><summary>Solution</summary>

```python
def ex4_batched_matmul(x: Tensor, y: Tensor) -> Tensor:
    return einsum(x, y, 'b i k, b k j -> b i j')
```
</details>

### Exercise 5 — attention scores QK^T (batched + reduce + matmul-like)

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Synthesize batched einsum with index-contraction to produce attention scores (QK^T) without reshaping.
> Keywords: attention, qkt, integration, multi-kc
> ```

**KCs targeted:** `einsum-matmul-contraction`, `einsum-batched`, `einsum-attention-scores`

Implement `ex5_attention_scores(q, k)` to compute pre-softmax attention scores.

Input shapes: `q` is `(b, q_len, d)`, `k` is `(b, k_len, d)`. Output shape: `(b, q_len, k_len)`. Each `out[b, i, j]` is the inner product `sum_d q[b, i, d] * k[b, j, d]`.

This is QK^T from a Transformer attention head, batched over `b`. Notice **you do not transpose** `k` — einsum handles the index alignment for you. `d` appears on both inputs and not on the output, so it's contracted.

> ⚠️ **Integrative exercise.** This combines 3+ KCs in one pattern; empirical work (Lohr et al. ITiCSE 2025) shows 3-concept LLM-generated exercises drop from ~94% to ~40% solvability. Expect a step in difficulty here vs Exercises 1-4.

In [ ]:
def ex5_attention_scores(q: Tensor, k: Tensor) -> Tensor:
    """Attention scores. (b, q_len, d) @ (b, k_len, d)^T → (b, q_len, k_len).

    Each out[b, i, j] = sum_d q[b, i, d] * k[b, j, d].
    """
    raise NotImplementedError()


def _test_ex5():
    b, q_len, k_len, d = 2, 3, 5, 4
    q = t.randn(b, q_len, d)
    k = t.randn(b, k_len, d)
    out = ex5_attention_scores(q, k)
    assert out.shape == (b, q_len, k_len), f'expected ({b},{q_len},{k_len}), got {out.shape}'
    # Ground truth via explicit batched matmul with manual transpose.
    expected = q @ k.transpose(-2, -1)
    assert t.allclose(out, expected, atol=1e-5), 'values differ from q @ k.transpose(-2,-1)'
    _dd_passed.add('ex5')
    print("ex5 ✓")

_test_ex5()

<details><summary>Solution</summary>

```python
def ex5_attention_scores(q: Tensor, k: Tensor) -> Tensor:
    return einsum(q, k, 'b q d, b k d -> b q k')
```

**Reading the pattern.**
- `b` appears in both inputs and on the output → carried through (batch).
- `q` only appears in the first input and on the output → preserved.
- `k` only appears in the second input and on the output → preserved.
- `d` appears in both inputs but **not** on the output → contracted (this is the dot product).

**Why this is the integrative case.** You're simultaneously batching (KC #4), preserving two independent non-contracted axes from different operands (extends KC #2 from `i,k → k,j` to a non-square layout), and letting omitting `d` do the reduction (KC #3). No transpose, no rearrange, no reshape — the pattern string carries the full intent.
</details>

## Done

Run the cell below to report your progress to Delta Drills. The beacon fires only if all 5 exercises passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1', 'ex2', 'ex3', 'ex4', 'ex5'}

def _dd_feedback_level(num_passed: int) -> str:
    """Map exercise-pass count → arena-rating feedback enum."""
    if num_passed == 5: return 'not_much'   # 5/5 → felt easy
    if num_passed >= 3: return 'somewhat'
    return 'a_lot'

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {len(missing)} exercises still failing: {sorted(missing)}.")
        print("[Delta Drills] not reporting until all 5 pass.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}',
        'subtopics': [DD_SUBTOPIC],
        'feedback': _dd_feedback_level(len(_dd_passed)),
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()